# 判别分析

### 1. 距离判别法

```sklearn.neighbors``` 模块的 ```KNeighborsClassifier``` 函数

```python
KneighborsClassifier(
    n_neighbors=5, 
    weights='uniform', 
    algorithm='auto', 
    leaf_size=20, 
    p=2, 
    metric='minkowski', 
    metric_params=None)
```
n_neighbors=5: 默认考虑 5 个最近邻  
weights='uniform': 默认对所有近邻赋予相同权重（均匀权重）  
algorithm='auto': 默认自动选择最合适的近邻搜索算法  
可以取 'ball_tree'
       'kd_tree'
       'brute'
leaf_size=20: 球树结构（如 BallTree、KDTree）的叶节点大小默认值  
p=2: Minkowski 距离的阶数，默认值 2 对应欧氏距离  
metric='minkowski': 默认使用 Minkowski 距离，倾向于使用Mahalanobis距离分类
metric_params=None: 距离 metric 的额外参数，默认无

以下是 scikit-learn 中 `KNeighborsClassifier` 类里 `algorithm` 和 `metric` 参数的可选值及含义说明: 

| 参数名   | 可选值                | 含义说明                                                                 |
|----------|-----------------------|--------------------------------------------------------------------------|
| `algorithm` | `'auto'`              | 自动根据数据类型和维度选择最合适的算法（默认值）                         |
|          | `'ball_tree'`         | 使用球树（BallTree）算法进行近邻搜索，适用于高维数据                     |
|          | `'kd_tree'`           | 使用KD树算法进行近邻搜索，适用于低维数据（通常比球树更快）               |
|          | `'brute'`             | 暴力搜索（线性扫描），适用于小数据集，无需构建树结构                     |
| `metric`   | `'minkowski'`         | 闵可夫斯基距离（默认值），是多种距离的泛化形式，通过 `p` 参数控制类型   |
|          | `'euclidean'`         | 欧氏距离（L2距离），等价于 `metric='minkowski'` 且 `p=2`                 |
|          | `'manhattan'`         | 曼哈顿距离（L1距离），等价于 `metric='minkowski'` 且 `p=1`               |
|          | `'chebyshev'`         | 切比雪夫距离，即两个样本各特征差值的最大值                               |
|          | `'mahalanobis'`       | 马氏距离，考虑特征间协方差的距离度量，需配合 `metric_params` 传入协方差矩阵 |
|          | `'hamming'`           | 汉明距离，用于二进制数据，计算不同特征的比例                             |
|          | `'cosine'`            | 余弦距离，衡量两个向量的夹角，常用于文本或高维数据                       |
|          | `'cityblock'`         | 城市街区距离，与曼哈顿距离等价                                           |

说明: 
- `algorithm` 的选择主要影响计算效率，小规模数据可选 `'brute'`，高维数据推荐 `'ball_tree'`
- 当 `metric='minkowski'` 时，`p=1` 对应曼哈顿距离，`p=2` 对应欧氏距离，`p→∞` 近似切比雪夫距离
- 部分距离（如 `'mahalanobis'`）需要通过 `metric_params` 参数传入额外信息

例11.1（1989年国际大学生数学建模竞赛A题: 蠓虫分类）虫是一种昆虫，分为很多类型，其中有一种名为Af，是能传播花粉的益虫；另一种名为Apf，是会传播疾病的害虫.这两种类型的虫在形态上十分相似，很难区别.现测得9只Af和6只Apf虫的触角长度和翅膀长度数据  

Af: (1.24, 1.27), (1.36, 1.74), (1.38, 1.64), (1.38, 1.82), (1.38, 1.90), (1.40, 1.70),(1.48, 1.82), (1.54, 1.82), (1.56, 2.08);

Apf: (1.14, 1.78), (1.18, 1.96), (1.20, 1.86), (1.26, 2.00), (1.28, 2.00), (1.30, 1.96).

若两类虫协方差矩阵相等，试判别（1.24，1.80），（1.28，1.84）与（1.40,2.04）3只
虫属于哪一类

In [18]:
import numpy as np
from sklearn.neighbors import KNeighborsClassifier  # 导入K近邻分类器 类(python)

# Af类（前9个）和Apf（后6个）类样本数据
x0 = np.array([[1.24, 1.27], [1.36, 1.74], [1.38, 1.64], 
               [1.38, 1.82], [1.38, 1.90], [1.40, 1.70], 
               [1.48, 1.82], [1.54, 1.82], [1.56, 2.08],
               [1.14, 1.78], [1.18, 1.96], [1.20, 1.86], 
               [1.26, 2.00], [1.28, 2.00], [1.30, 1.96]])  # (15, 2)
# 样本数据
x = np.array([[1.24, 1.80], [1.28, 1.84], [1.40, 2.04]])
# 两类样本数据的编号，Af编号为1，Apf编号为2
g = np.hstack([np.ones(9), 2 * np.ones(6)])  # g = [1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2]
# np.cov(m, rowvar=True)即每行是一个变量,先转置成(2, 15),第一行是触角长度，第二行是翅膀长度
v = np.cov(x0.T)  # 计算协方差矩阵

# metric距离方式，返回值为 KNeighborsClassifier 实例对象 
knn = KNeighborsClassifier(n_neighbors=2, metric='mahalanobis', metric_params={'V': v})  # 用马氏距离分类
knn.fit(x0, g)  # 无返回值，直接原地修改对象，把训练信息储存到knn内部
pre = knn.predict(x)  # 预测，返回array数组，dtype与g一致，长度为x的列数

print(f"马氏距离分类结果:{pre}")
print(f"马氏距离已知样本的误判率为: {1 - knn.score(x0, g)}")

knn2 = KNeighborsClassifier(2)  # 用欧氏距离分类
knn2.fit(x0, g)
pre2 = knn2.predict(x)

print(f"欧氏距离分类结果: {pre2}")
print(f"欧氏距离已知样本的误判率为: {1 - knn2.score(x0, g)}")


马氏距离分类结果:[2. 2. 1.]
马氏距离已知样本的误判率为: 0.0
欧氏距离分类结果: [2. 1. 2.]
欧氏距离已知样本的误判率为: 0.0


从健康人群、硬化症患者和冠心病患者中分别随机选取10人、 6人和 4 人，考察了他们各自心电图的 5 个不同指标（记作 $x_1, x_2, x_3, x_4, x_5$）如表 11.2所示，试对两个待判样品作出判断．

In [24]:
import pandas as pd
from IPython.display import display

df = pd.read_excel("Pdata11_2.xlsx", header=None)  # 没有表头，即列名
df.columns = ['序号', 'x1', 'x2', 'x3', 'x4', 'x5', '类型']
display(df.style.hide(axis="index"))  # 隐藏自带的行索引

序号,x1,x2,x3,x4,x5,类型
1,8.110000,261.010000,13.230000,5.460000,7.360000,1
2,9.360000,185.390000,9.020000,5.660000,5.990000,1
3,9.850000,249.580000,15.610000,6.060000,6.110000,1
4,2.550000,137.130000,9.210000,6.110000,4.350000,1
5,6.010000,231.340000,14.270000,5.210000,8.790000,1
6,9.460000,231.380000,13.030000,4.880000,8.530000,1
7,4.110000,260.250000,14.720000,5.360000,10.020000,1
8,8.900000,259.510000,14.160000,4.910000,9.790000,1
9,7.710000,273.840000,16.010000,5.150000,8.790000,1
10,7.510000,303.590000,19.140000,5.700000,8.530000,1


In [37]:
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier  # scikit learn"scientific"（科学的）和"kit"（工具包）

df = pd.read_excel("Pdata11_2.xlsx", header=None)
val = df.values  # 转换成array数组，汉字转换为字符串
x0 = val[:-2,1:-1].astype(float)  # 提取已知样本点的观测值，最后两行待测不取，第一列序号和最后一列标签不取
y0 = val[:-2, -1].astype(int)  # 取个样本的标签（类）
x = val[[-2, -1], 1:-1]  # 提取待测样本点的观测值

v = np.cov(x0.T)
knn = KNeighborsClassifier(3, metric='mahalanobis', metric_params={'V': v})
knn.fit(x0, y0)  # knn.fit(sample, label)
pre = knn.predict(x)

"""
误判率比较: (n_neighbors=3)
欧氏距离(euclidean): [1, 3]  19.999999999999996%
曼哈顿距离(manhattan): [1, 3]  19.999999999999996%
切比雪夫距离(chebyshev): [1, 3]  19.999999999999996%
马氏距离(mahalanobis): [1, 1]  15.000000000000002%
可以看到马氏距离误判率最低
"""

print(f"分类结果: {pre}")
print(f"已知样本的误判率: {1 - knn.score(x0, y0)}")


分类结果: [1 1]
已知样本的误判率: 0.15000000000000002


在 Python 的 `scikit-learn` 库中，`LinearDiscriminantAnalysis`（LDA）是实现线性判别分析的核心类，主要用于降维和分类任务。以下是其调用格式、核心参数、返回值的详细说明：


### **一、调用格式**
首先需要从 `sklearn.discriminant_analysis` 导入类，然后实例化并使用：
```python
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis（线性判别分析，即fisher linear discrinant（判别） analysis）

# 1. 实例化模型（设置参数）
lda = LinearDiscriminantAnalysis(**参数**)

# 2. 训练模型（拟合数据）
lda.fit(X, y)

# 3. 预测（分类任务）或降维（转换数据）
y_pred = lda.predict(X_new)  # 分类预测
X_reduced = lda.transform(X)  # 降维转换
```


### **二、核心参数及含义**
`LinearDiscriminantAnalysis` 的常用参数如下（按重要性排序）：

| 参数名 | 类型 | 含义 |
|--------|------|------|
| `solver` | str | 求解器类型，决定LDA的计算方式。<br>可选值：<br>- `"svd"`（默认）：奇异值分解，适用于大数据集，不计算类内散布矩阵的逆，数值稳定性好。<br>- `"lsqr"`：最小二乘解，支持正则化（需配合 `shrinkage` 参数）。<br>- `"eigen"`：特征值分解，结合正则化（需配合 `shrinkage`），适用于高维小样本。 |
| `shrinkage` | str / float / None | 正则化参数，用于解决高维数据中类内散布矩阵可能奇异（不可逆）的问题。<br>可选值：<br>- `None`（默认）：不正则化。<br>- `"auto"`：自动选择最优正则化参数（仅 `solver="lsqr"` 或 `"eigen"` 支持）。<br>- 0~1的浮点数：手动设置正则化强度（值越大，正则化越强）。 |
| `n_components` | int / None | 降维后的维度（仅用于降维任务）。<br>取值范围：最多为 `min(n_classes - 1, n_features)`（类别数-1或特征数，取较小值）。<br>默认 `None`：不主动降维（保留所有有效维度）。 |
| `priors` | array-like of shape (n_classes,) / None | 类别先验概率（即每个类别的占比）。<br>- `None`（默认）：从训练数据中自动估计（样本比例）。<br>- 手动指定：需满足元素和为1，例如 `[0.3, 0.7]` 表示两类的先验概率分别为30%和70%。 |
| `store_covariance` | bool | 是否存储类内协方差矩阵。<br>- `False`（默认）：不存储。<br>- `True`：存储（可通过 `covariance_` 属性获取）。 |
| `tol` | float | 收敛阈值（仅 `solver="lsqr"` 时有效），默认 `1e-4`。 |


### **三、返回值（模型属性）**
训练后的 `LinearDiscriminantAnalysis` 实例包含多个属性，反映模型的核心计算结果：

| 属性名 | 类型 | 含义 |
|--------|------|------|
| `coef_` | array-like of shape (n_classes-1, n_features) | 判别系数（仅二分类时直观）。<br>对于二分类，形状为 `(1, n_features)`，表示投影系数（类似权重）；多分类时为 `(n_classes-1, n_features)`，对应多个判别函数。 |
| `intercept_` | array-like of shape (n_classes-1,) | 判别函数的截距项，与 `coef_` 共同构成线性判别函数：`decision_function(X) = X @ coef_.T + intercept_`。 |
| `scalings_` | array-like of shape (n_features, n_components) | 降维转换的缩放矩阵。<br>数据降维时，`X_reduced = X @ scalings_`（即通过此矩阵将原始特征投影到低维空间）。 |
| `means_` | array-like of shape (n_classes, n_features) | 每个类别的均值向量，反映训练集中各类别的中心位置。 |
| `covariance_` | array-like of shape (n_features, n_features) | 类内协方差矩阵（仅当 `store_covariance=True` 时可用），即总类内散布矩阵除以样本量的估计。 |
| `explained_variance_ratio_` | array-like of shape (n_components,) | 每个维度解释的方差比例（类似PCA的方差解释率），反映降维后各维度的重要性，总和为1。 |
| `classes_` | array-like of shape (n_classes,) | 训练数据中的类别标签（如分类任务中的 `[0, 1, 2]`）。 |
| `priors_` | array-like of shape (n_classes,) | 实际使用的类别先验概率（若未手动指定，则为训练数据中的类别比例）。 |


### **四、常用方法**
除了上述 `fit()`、`predict()`、`transform()` 外，还有几个实用方法：

| 方法 | 作用 |
|------|------|
| `decision_function(X)` | 返回样本在判别函数上的投影值（用于分类的置信度），形状为 `(n_samples, n_classes-1)`。 |
| `predict_proba(X)` | 返回每个样本属于各类别的概率，形状为 `(n_samples, n_classes)`，适用于需要概率输出的场景。 |
| `score(X, y)` | 计算模型在测试集上的准确率（分类任务）。 |


### **总结**
`LinearDiscriminantAnalysis` 是一个灵活的工具，通过调整 `solver` 和 `n_components` 可兼顾分类和降维任务。核心参数 `solver` 影响计算效率和稳定性，`n_components` 控制降维后的维度，`shrinkage` 用于高维数据的正则化。训练后通过 `coef_`、`scalings_` 等属性可深入理解模型的判别逻辑和降维原理。

In [38]:
# 蠓虫
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA

# Af类（前9个）和Apf（后6个）类样本数据
x0 = np.array([[1.24, 1.27], [1.36, 1.74], [1.38, 1.64], 
               [1.38, 1.82], [1.38, 1.90], [1.40, 1.70], 
               [1.48, 1.82], [1.54, 1.82], [1.56, 2.08],
               [1.14, 1.78], [1.18, 1.96], [1.20, 1.86], 
               [1.26, 2.00], [1.28, 2.00], [1.30, 1.96]])  # (15, 2)
# 待测样本数据
x = np.array([[1.24, 1.80], [1.28, 1.84], [1.40, 2.04]])
# 两类样本数据的编号(类别/标签)，Af编号为1，Apf编号为2
y0 = np.hstack([np.ones(9), 2 * np.ones(6)])

clf = LDA()
clf.fit(x0, y0)

print(f"判别结果为: {clf.predict(x)}")
print(f"误判率为: {1 - clf.score(x0, y0)}")

判别结果为: [2. 2. 2.]
误判率为: 0.0


In [39]:
# 心脏病
import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier  # scikit learn"scientific"（科学的）和"kit"（工具包）

df = pd.read_excel("Pdata11_2.xlsx", header=None)
val = df.values  # 转换成array数组，汉字转换为字符串
x0 = val[:-2,1:-1].astype(float)  # 提取已知样本点的观测值，最后两行待测不取，第一列序号和最后一列标签不取
y0 = val[:-2, -1].astype(int)  # 取个样本的标签（类）
x = val[[-2, -1], 1:-1]  # 提取待测样本点的观测值

clf = LDA()  # clf即classfier分类器， LDA 模型对象，若用降维，可命名为ida
clf.fit(x0, y0)

print(f"判别结果为: {clf.predict(x)}")
print(f"误判率为: {1 - clf.score(x0, y0)}")

判别结果为: [1 2]
误判率为: 0.0


在 `scikit-learn` 库中，`GaussianNB` 是实现**高斯朴素贝叶斯（Gaussian Naive Bayes）** 算法的类，专门用于分类任务。它是贝叶斯判别法在“假设特征服从高斯分布”和“特征条件独立”下的简化实现，以下从调用格式、核心参数、属性及方法等方面详细介绍：


### **一、调用格式**
使用前需从 `sklearn.naive_bayes` 导入类，基本流程为“实例化模型→训练→预测”：
```python
from sklearn.naive_bayes import GaussianNB

# 1. 实例化模型（可配置参数）
gnb = GaussianNB(**参数**)

# 2. 训练模型（拟合数据）
gnb.fit(X, y)

# 3. 预测新样本类别
y_pred = gnb.predict(X_new)
```


### **二、核心参数及含义**
`GaussianNB` 的参数较少，主要用于控制数值稳定性，核心参数如下：

| 参数名 | 类型 | 含义 |
|--------|------|------|
| `priors` | array-like of shape (n_classes,) / None | 类别先验概率。<br>- `None`（默认）：从训练数据中自动估计（按类别样本占比计算）。<br>- 手动指定：需传入与类别数量相同的数组，元素之和为1（如二分类传入 `[0.3, 0.7]`）。 |
| `var_smoothing` | float | 用于数值稳定性的正则化参数，默认 `1e-9`。<br>作用：将所有特征的方差加上一个极小值（`var_smoothing`），避免因某特征方差为0导致的计算错误（如除以零）。<br>值越大，正则化越强（但可能降低模型精度）。 |


### **三、模型属性（训练后可用）**
训练后的 `GaussianNB` 实例包含以下核心属性，反映模型学习到的关键信息：

| 属性名 | 类型 | 含义 |
|--------|------|------|
| `class_prior_` | array-like of shape (n_classes,) | 实际使用的类别先验概率。<br>若训练时指定了 `priors`，则直接等于该值；否则为训练集中各类别样本的占比。 |
| `class_count_` | array-like of shape (n_classes,) | 训练集中每个类别包含的样本数量（整数）。 |
| `theta_` | array-like of shape (n_classes, n_features) | 每个类别中各特征的均值（对应高斯分布的均值 \( \mu \)）。<br>例如 `theta_[i, j]` 表示第 \( i \) 类中第 \( j \) 个特征的均值。 |
| `sigma_` | array-like of shape (n_classes, n_features) | 每个类别中各特征的方差（对应高斯分布的方差 \( \sigma^2 \)）。<br>例如 `sigma_[i, j]` 表示第 \( i \) 类中第 \( j \) 个特征的方差（已加上 `var_smoothing` 确保稳定性）。 |
| `classes_` | array-like of shape (n_classes,) | 训练数据中的类别标签（如 `[0, 1, 2]`）。 |


### **四、常用方法**
`GaussianNB` 提供了与 `scikit-learn` 其他模型统一的接口方法：

| 方法 | 作用 | 返回值类型 |
|------|------|------------|
| `fit(X, y)` | 训练模型，学习每个类别的均值、方差和先验概率。 | 模型本身（`self`） |
| `predict(X)` | 预测输入样本的类别。 | 数组（`shape=(n_samples,)`），每个元素为预测的类别标签 |
| `predict_proba(X)` | 预测输入样本属于每个类别的概率（后验概率）。 | 数组（`shape=(n_samples, n_classes)`），每行之和为1 |
| `predict_log_proba(X)` | 预测输入样本属于每个类别的对数后验概率（数值稳定性更好）。 | 数组（`shape=(n_samples, n_classes)`） |
| `score(X, y)` | 计算模型在输入数据上的分类准确率（正确预测数/总样本数）。 | 浮点数（0~1之间） |


### **五、核心原理与“朴素”的含义**
`GaussianNB` 是贝叶斯判别法的简化版，核心假设体现为“朴素”（Naive）：

1. **高斯分布假设**：  
   假设每个类别中，特征服从**高斯（正态）分布**，即类条件概率 \( P(x_j \mid \omega_i) \sim \mathcal{N}(\mu_{i,j}, \sigma^2_{i,j}) \)，其中 \( \mu_{i,j} \) 是第 \( i \) 类第 \( j \) 个特征的均值（`theta_[i,j]`），\( \sigma^2_{i,j} \) 是对应的方差（`sigma_[i,j]`）。

2. **特征条件独立假设**：  
   这是“朴素”的核心——假设同一类别下，不同特征之间**相互独立**。因此，联合概率可简化为各特征概率的乘积：  
   \[
   P(x \mid \omega_i) = \prod_{j=1}^n P(x_j \mid \omega_i)
   \]  
   这一假设极大降低了计算复杂度（无需计算协方差矩阵），但也牺牲了特征间的关联性信息。


### **六、适用场景与优缺点**
- **优点**：  
  1. 计算高效（仅需计算均值和方差），适合高维数据和大数据集。  
  2. 对缺失数据不敏感，对噪声鲁棒性较强。  
  3. 需少量训练样本即可收敛（尤其当特征独立假设成立时）。  

- **缺点**：  
  1. 特征条件独立假设在现实中往往不成立（如身高和体重存在相关性），可能影响精度。  
  2. 对特征分布的高斯假设依赖性强，若实际分布偏离高斯分布较远，性能会下降。  

- **典型应用**：  
  文本分类（如垃圾邮件识别）、情感分析、简单的图像分类等场景，尤其适合作为“ baseline（基准模型）”快速验证思路。


### **总结**
`GaussianNB` 是贝叶斯判别法的轻量实现，通过“高斯分布+特征独立”假设简化计算，以牺牲部分精度换取高效性。其核心参数 `var_smoothing` 用于保障数值稳定，训练后可通过 `theta_` 和 `sigma_` 查看每个类别下的特征分布规律，是处理分类问题的实用工具。

In [40]:
# 蠓虫
import numpy as np
from sklearn.naive_bayes import GaussianNB

# Af类（前9个）和Apf（后6个）类样本数据
x0 = np.array([[1.24, 1.27], [1.36, 1.74], [1.38, 1.64], 
               [1.38, 1.82], [1.38, 1.90], [1.40, 1.70], 
               [1.48, 1.82], [1.54, 1.82], [1.56, 2.08],
               [1.14, 1.78], [1.18, 1.96], [1.20, 1.86], 
               [1.26, 2.00], [1.28, 2.00], [1.30, 1.96]])  # (15, 2)
# 待测样本数据
x = np.array([[1.24, 1.80], [1.28, 1.84], [1.40, 2.04]])
# 两类样本数据的编号(类别/标签)，Af编号为1，Apf编号为2
y0 = np.hstack([np.ones(9), 2 * np.ones(6)])

clf = GaussianNB()
clf.fit(x0, y0)

print(f"判别结果为: {clf.predict(x)}")
print(f"已知样本的误判率为: {1 - clf.score(x0, y0)}")

判别结果为: [2. 2. 1.]
已知样本的误判率为: 0.0


### 判断准则的评价

sklearn 库中的 ```sklearn.model_selection``` 模块的 ```cross_cal_score``` （cross validaton交叉验证）函数可以计算交叉检验的精度，其调用格式为：

```cross_val_score(model, x0, y0, cv = k)```

其中，models: 建立的模型  
          x0: 已知样本点的数据  
          y0: 已知样本的编号  
          cv = k: 表示把已知样本点分成 $k$ 组，其中 $k-1$ 组被用作训练集，剩下一组被用作验证集，这样一共可以对分类器做 $k$ 次训练，并且得到 $k$ 个训练结果  

返回值: 每组评估数据分类的准确率  

In [41]:
import numpy as np
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import cross_val_score

df = pd.read_excel("Pdata11_2.xlsx", header=None)
b = df.values

x0 = b[:-2, 1:-1].astype(float)
y0 = b[:-2, -1].astype(int)

model = LinearDiscriminantAnalysis()

print(cross_val_score(model, x0, y0, cv = 2))
# [0.9 0.8]两组数据的准确率分别为 90% 和 80%

[0.9 0.8]
